In [1]:
# --- Portable data location (added by macos-native/tools/mac_python_paths.py) ---------------------
# RAMAN_HANDOFF_DIR is the hand-off folder laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug,
# found by walking up from this file (a notebook: from its working folder). To use another copy,
# set the RAMAN_HANDOFF_DIR environment variable to that folder.
import os as _os


def _raman_handoff_dir():
    if _os.environ.get("RAMAN_HANDOFF_DIR"):
        chosen = _os.path.abspath(_os.path.expanduser(_os.environ["RAMAN_HANDOFF_DIR"]))
        if not _os.path.isdir(chosen):
            raise FileNotFoundError("RAMAN_HANDOFF_DIR is set to " + chosen + ", which is not a folder")
        return chosen
    start = _os.path.dirname(_os.path.abspath(globals()["__file__"])) if "__file__" in globals() else _os.getcwd()
    folder = start
    while True:
        candidate = _os.path.join(folder, "laser_test", "build", "Desktop_Qt_6_6_3_MSVC2019_64bit-Debug")
        if _os.path.isdir(candidate):
            return candidate
        parent = _os.path.dirname(folder)
        if parent == folder:
            raise FileNotFoundError("laser_test/build/Desktop_Qt_6_6_3_MSVC2019_64bit-Debug not found above "
                                    + start + "; set the RAMAN_HANDOFF_DIR environment variable")
        folder = parent


RAMAN_HANDOFF_DIR = _raman_handoff_dir()
# ---------------------------------------------------------------------------------------------------
"""
ViT-1D Raman classifier (Vision Transformer 1D)
"""

import os
import sys
import re
import json
import time
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix

# =========================================================
# CONFIG
# =========================================================
DATA_ROOT = RAMAN_HANDOFF_DIR

# ---- Training data: read .npy files made by v5 preprocessing (same as PCA-SVM) ----
TRAINING_NPY_FILES = {
    "r6g": os.path.join(DATA_ROOT, "data_base_code_part", "training_spectra_r6g.npy"),
    "methylene_blue": os.path.join(DATA_ROOT, "data_base_code_part", "training_spectra_methylene_blue.npy"),
}

# ---- Test file: still raw log, will be preprocessed at runtime ----
TEST_FILE = f"{RAMAN_HANDOFF_DIR}/Portable_Raman_database/Database_mythylene_blue.txt"
TEST_DARK_FILE = f"{RAMAN_HANDOFF_DIR}/Portable_Raman_database/dark_mythylene_blue.txt"

# ---- Train again or just load? ----
RETRAIN_MODEL = True

# ---- Preprocessing settings (must match v5 exactly) ----
DROP_N = 30
TARGET_LEN_RAW = 2048              # raw spectrum length
TARGET_LEN_AFTER_DROP = 2018       # length after dark sub + drop 30
TARGET_LEN = 2048                  # we pad the 2018-length feature vector to 2048
PATCH_SIZE = 64                    # 2048 / 64 = 32 tokens per spectrum
NUM_PATCHES = TARGET_LEN // PATCH_SIZE  # 32

DO_BASELINE = True
BASELINE_WINDOW = 15
BASELINE_SMOOTH = 15

# ---- Quality filter at inference time ----
ENABLE_QUALITY_FILTER = True
RAW_MIN_PTP = 100.0
RAW_MIN_STD = 15.0

# ---- Training settings ----
BATCH_SIZE = 32
EPOCHS = 100             # ViT usually needs more epochs than CNN
LR = 5e-4                # ViT is sensitive to LR; lower is safer
WEIGHT_DECAY = 1e-4
SEED = 42
WARMUP_EPOCHS = 5        # ViT standard: linear warmup then cosine decay

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20
NUM_WORKERS = 0

# ---- Data augmentation ----
USE_AUGMENT = True
AUG_NOISE_STD = 0.02
AUG_SHIFT_MAX = 10
AUG_SCALE_LOW = 0.95
AUG_SCALE_HIGH = 1.05
AUG_BASELINE_DRIFT = 0.12

# ---- Model / loss ----
D_MODEL = 128            # transformer hidden size (token embedding dim)
NUM_HEADS = 4            # multi-head attention heads (D_MODEL must be divisible)
DEPTH = 4                # number of transformer blocks
MLP_RATIO = 4.0          # FFN hidden = D_MODEL * MLP_RATIO
DROPOUT = 0.15           # dropout in attention / FFN
ATTN_DROP = 0.10         # attention weight dropout
DROP_PATH = 0.10         # stochastic depth (per-block residual drop)
LABEL_SMOOTHING = 0.05
GRAD_CLIP_NORM = 3.0
EMBED_DIM = D_MODEL      # CLS token output dim, used for centroid/distance OOD

# ---- Open-world rejection (4 gates) ----
MIN_BLOCK_CONF = 0.60
MIN_BLOCK_MARGIN = 0.15
MIN_ACCEPT_BLOCKS = 3
MIN_ACCEPT_RATIO = 0.20
MIN_CLASS_CONSENSUS = 0.75
INFER_TEMPERATURE = 1.6

# ---- Output ----
OUTPUT_DIR = os.path.join(DATA_ROOT, "_ViT1D_raman_v1")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
MODEL_PATH = os.path.join(MODELS_DIR, "vit1d_raman.pt")
META_PATH = os.path.join(OUTPUT_DIR, "meta.json")
CENTROIDS_PATH = os.path.join(OUTPUT_DIR, "centroids.npz")
os.makedirs(MODELS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = DEVICE.startswith("cuda")
print("DEVICE =", DEVICE)
print("MODEL_PATH =", MODEL_PATH)
print("RETRAIN_MODEL =", RETRAIN_MODEL)
print(f"Model: ViT-1D  d_model={D_MODEL}  heads={NUM_HEADS}  depth={DEPTH}  "
      f"patch_size={PATCH_SIZE}  num_tokens={NUM_PATCHES}")


# =========================================================
# UTILS
# =========================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


def _is_jupyter() -> bool:
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell in ("ZMQInteractiveShell", "TerminalInteractiveShell"):
            return True
    except Exception:
        pass
    return len(sys.argv) > 0 and "jupyter" in sys.argv[0].lower()


def _parse_cli_args() -> Tuple[str, str]:
    if _is_jupyter():
        return TEST_FILE, TEST_DARK_FILE
    args = [a for a in sys.argv[1:] if not a.startswith("-")]
    tf = args[0].strip() if len(args) >= 1 and args[0].strip() else TEST_FILE
    td = args[1].strip() if len(args) >= 2 and args[1].strip() else TEST_DARK_FILE
    return tf, td


def softmax_np(logits: np.ndarray) -> np.ndarray:
    z = np.asarray(logits, np.float64)
    if z.size == 0 or (not np.isfinite(z).all()):
        return np.zeros_like(z, np.float32)
    z = z - z.max()
    e = np.exp(z)
    s = e.sum()
    if s <= 0 or (not np.isfinite(s)):
        return np.zeros_like(z, np.float32)
    return (e / s).astype(np.float32)


def percentile_safe(values: List[float], q: float, fallback: float) -> float:
    if not values:
        return float(fallback)
    arr = np.asarray(values, np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float(fallback)
    return float(np.percentile(arr, q))


def pad_or_trim_to(x: np.ndarray, target_len: int) -> np.ndarray:
    """Pad with zeros (right) or trim to fixed length so it divides patch_size."""
    x = np.asarray(x, dtype=np.float32)
    if x.size == target_len:
        return x
    if x.size > target_len:
        return x[:target_len].astype(np.float32)
    out = np.zeros(target_len, dtype=np.float32)
    out[:x.size] = x
    return out


# =========================================================
# Preprocessing functions used at inference time (same logic as v5)
# =========================================================
ERROR_PHRASE = "image data error code 0x0001"


def strip_prefix(line: str) -> str:
    line = line.strip()
    line = re.sub(r"^\[[^\]]+\]\s+\[Backend\]\s*", "", line)
    line = re.sub(r"^\[[^\]]+\]\s*", "", line)
    return line.strip()


def load_dark_spectrum_from_file(dark_path: str) -> np.ndarray:
    if not dark_path or not Path(dark_path).exists():
        return np.array([], dtype=np.float64)
    intensities: List[float] = []
    with open(dark_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or line.startswith("index"):
                continue
            parts = line.split("\t")
            if len(parts) >= 3:
                try:
                    intensities.append(float(parts[2]))
                except ValueError:
                    continue
    return np.array(intensities, dtype=np.float64)


def parse_portable_log_to_spectra(log_file: str) -> Tuple[List[np.ndarray], int]:
    lines = open(log_file, "r", encoding="utf-8", errors="ignore").read().splitlines()
    spectra: List[np.ndarray] = []
    failed = 0
    i = 0
    while i < len(lines):
        line_raw = lines[i]
        line = strip_prefix(line_raw)
        if ERROR_PHRASE in line_raw.lower():
            failed += 1
            i += 1
            continue
        if line.startswith("Image Data"):
            tokens = list(re.findall(r"\b[0-9A-Fa-f]{2,4}\b", line[len("Image Data"):].strip()))
            j = i + 1
            block_err = False
            while j < len(lines):
                nxt_raw = lines[j]
                nxt = strip_prefix(nxt_raw)
                if ERROR_PHRASE in nxt_raw.lower():
                    block_err = True
                if nxt.startswith("Stop Capture") or (nxt.startswith("Image Data") and j != i):
                    break
                hexes = re.findall(r"\b[0-9A-Fa-f]{2,4}\b", nxt)
                if hexes:
                    tokens.extend(hexes)
                j += 1
            if block_err or not tokens:
                failed += 1
            else:
                vals = np.array([int(t, 16) for t in tokens], dtype=np.int64)
                if vals.size % 2 == 0 and vals.max(initial=0) <= 255:
                    le = vals[0::2] + (vals[1::2] << 8)
                    be = (vals[0::2] << 8) + vals[1::2]
                    v16 = le if le.max(initial=0) >= be.max(initial=0) else be
                    if v16.max(initial=0) > 255:
                        vals = v16.astype(np.int64)
                spectra.append(vals.astype(np.float32))
            i = j + 1
            continue
        i += 1
    return spectra, failed


def moving_average(y: np.ndarray, window: int) -> np.ndarray:
    window = int(max(3, window))
    if window % 2 == 0:
        window += 1
    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / window
    return np.convolve(ypad, kernel, mode="valid")


def rolling_minimum(y: np.ndarray, window: int) -> np.ndarray:
    window = int(max(3, window))
    if window % 2 == 0:
        window += 1
    pad = window // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    out = np.empty_like(y, dtype=np.float64)
    for i in range(y.size):
        out[i] = np.min(ypad[i:i + window])
    return out


def baseline_correction_v5(y: np.ndarray, window: int = 15, smooth: int = 15) -> np.ndarray:
    y = np.asarray(y, dtype=np.float64)
    if y.size < 5:
        return y.copy().astype(np.float32)
    baseline = rolling_minimum(y, window=window)
    baseline = moving_average(baseline, window=smooth)
    return (y - baseline).astype(np.float32)


def normalize_min_max(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float64)
    ymin, ymax = float(np.min(y)), float(np.max(y))
    denom = ymax - ymin
    if denom < 1e-12:
        return np.zeros_like(y, dtype=np.float32)
    return ((y - ymin) / denom).astype(np.float32)


def preprocess_single_spectrum_v5(raw_spectrum: np.ndarray, dark_raw: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
    s = np.asarray(raw_spectrum, np.float64)
    if s.size < TARGET_LEN_RAW:
        return None, None
    s2048 = s[:TARGET_LEN_RAW]
    d2048 = dark_raw[:TARGET_LEN_RAW] if dark_raw.size >= TARGET_LEN_RAW else dark_raw
    if d2048.size >= TARGET_LEN_RAW:
        raw_dark = s2048 - d2048
    else:
        raw_dark = s2048.copy()
    if DROP_N >= raw_dark.size:
        return None, None
    raw_dark = raw_dark[DROP_N:]   # length 2018
    quality = {
        "ptp": float(np.ptp(raw_dark)),
        "std": float(np.std(raw_dark)),
    }
    if DO_BASELINE:
        corrected = baseline_correction_v5(raw_dark, window=BASELINE_WINDOW, smooth=BASELINE_SMOOTH)
    else:
        corrected = raw_dark.astype(np.float32)
    normed = normalize_min_max(corrected)
    final = pad_or_trim_to(normed, TARGET_LEN)
    return final.astype(np.float32), quality


# =========================================================
# STEP 1: Load preprocessed training data (.npy)
# =========================================================
class_names = list(TRAINING_NPY_FILES.keys())
class_to_id = {c: i for i, c in enumerate(class_names)}
id_to_class = {i: c for c, i in class_to_id.items()}
NUM_CLASSES = len(class_names)

print("\n" + "=" * 70)
print("Step 1: Load pre-processed training data from .npy files")
print("=" * 70)

X_parts: List[np.ndarray] = []
Y_parts: List[np.ndarray] = []

for cls, npy_path in TRAINING_NPY_FILES.items():
    if not os.path.exists(npy_path):
        raise FileNotFoundError(f"Cannot find {cls} training file: {npy_path}")
    data = np.load(npy_path).astype(np.float32)
    print(f"  {cls:20s}  shape={data.shape}")
    if data.shape[1] != TARGET_LEN_AFTER_DROP:
        raise ValueError(f".npy feature length is {data.shape[1]}, expected {TARGET_LEN_AFTER_DROP}")
    padded = np.zeros((data.shape[0], TARGET_LEN), dtype=np.float32)
    padded[:, :TARGET_LEN_AFTER_DROP] = data
    X_parts.append(padded)
    Y_parts.append(np.full(data.shape[0], class_to_id[cls], dtype=np.int64))

X = np.vstack(X_parts).astype(np.float32)
Y = np.concatenate(Y_parts)
FEATURE_DIM = X.shape[1]

print(f"\nTotal: {X.shape}  (padded from 2018 to {TARGET_LEN})")
print(f"Classes: {dict((id_to_class[i], int((Y==i).sum())) for i in range(NUM_CLASSES))}")
print(f"Value range: [{X.min():.4f}, {X.max():.4f}]")


# =========================================================
# STEP 2: Train / Val / Test split (stratified)
# =========================================================
def stratified_split_indices_3way(Y, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                                  test_ratio=TEST_RATIO, seed=SEED):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    rng = random.Random(seed)
    idx_tr, idx_va, idx_te = [], [], []
    for cls in np.unique(Y):
        idx = np.where(Y == cls)[0].tolist()
        rng.shuffle(idx)
        n = len(idx)
        n_tr = max(1, int(round(n * train_ratio)))
        n_va = max(1, int(round(n * val_ratio)))
        if n_tr + n_va >= n:
            n_va = max(1, n - n_tr - 1)
        if n_tr + n_va >= n:
            n_tr = max(1, n - 2)
            n_va = 1
        idx_tr.extend(idx[:n_tr])
        idx_va.extend(idx[n_tr:n_tr + n_va])
        idx_te.extend(idx[n_tr + n_va:])
    rng.shuffle(idx_tr)
    rng.shuffle(idx_va)
    rng.shuffle(idx_te)
    return idx_tr, idx_va, idx_te


idx_tr, idx_va, idx_te = stratified_split_indices_3way(Y)
print(f"\nSplit ratio: train={TRAIN_RATIO}  val={VAL_RATIO}  test={TEST_RATIO}")
print(f"Split count: train={len(idx_tr)}  val={len(idx_va)}  test={len(idx_te)}")
for i in range(NUM_CLASSES):
    n_tr = sum(1 for k in idx_tr if Y[k] == i)
    n_va = sum(1 for k in idx_va if Y[k] == i)
    n_te = sum(1 for k in idx_te if Y[k] == i)
    print(f"  {id_to_class[i]:20s}  train={n_tr}  val={n_va}  test={n_te}")


# =========================================================
# DATASET (with augmentation)
# =========================================================
def augment_spectrum_1d(x: np.ndarray) -> np.ndarray:
    x = x.copy().astype(np.float32)
    x *= np.float32(np.random.uniform(AUG_SCALE_LOW, AUG_SCALE_HIGH))
    shift = int(np.random.randint(-AUG_SHIFT_MAX, AUG_SHIFT_MAX + 1))
    if shift:
        x = np.roll(x, shift)
    drift = np.linspace(-1, 1, x.size, dtype=np.float32)
    x = x + np.random.normal(0, AUG_NOISE_STD, x.shape).astype(np.float32)
    x = x + np.float32(np.random.uniform(-AUG_BASELINE_DRIFT, AUG_BASELINE_DRIFT)) * drift
    return normalize_min_max(x)


class RamanDataset(Dataset):
    def __init__(self, X, Y, indices, augment=False):
        self.X = X
        self.Y = Y
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        x = self.X[i].copy()
        if self.augment:
            x = augment_spectrum_1d(x)
        return torch.from_numpy(x), torch.tensor(int(self.Y[i]), dtype=torch.long)


train_loader = DataLoader(RamanDataset(X, Y, idx_tr, USE_AUGMENT), BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(RamanDataset(X, Y, idx_va, False), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(RamanDataset(X, Y, idx_te, False), BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# =========================================================
# MODEL: 1D Vision Transformer (ViT-1D)
# =========================================================
class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.rand(shape, dtype=x.dtype, device=x.device)
        mask = torch.floor(mask + keep)
        return x * mask / keep


class PatchEmbedding1D(nn.Module):
    def __init__(self, seq_len: int, patch_size: int, d_model: int):
        super().__init__()
        assert seq_len % patch_size == 0, \
            f"seq_len ({seq_len}) must be divisible by patch_size ({patch_size})"
        self.patch_size = patch_size
        self.num_patches = seq_len // patch_size
        self.proj = nn.Linear(patch_size, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L = x.shape
        x = x.view(B, self.num_patches, self.patch_size)
        x = self.proj(x)
        x = self.norm(x)
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, attn_drop: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(attn_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        return out


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, mlp_ratio: float = 4.0,
                 dropout: float = 0.0, attn_drop: float = 0.0, drop_path: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, num_heads, attn_drop)
        self.dp1 = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )
        self.dp2 = DropPath(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dp1(self.attn(self.norm1(x)))
        x = x + self.dp2(self.ffn(self.norm2(x)))
        return x


class RamanViT1D(nn.Module):
    def __init__(self, seq_len=TARGET_LEN, patch_size=PATCH_SIZE, d_model=D_MODEL,
                 num_heads=NUM_HEADS, depth=DEPTH, mlp_ratio=MLP_RATIO,
                 num_classes=NUM_CLASSES, dropout=DROPOUT, attn_drop=ATTN_DROP,
                 drop_path=DROP_PATH):
        super().__init__()
        self.d_model = d_model
        self.patch_embed = PatchEmbedding1D(seq_len, patch_size, d_model)
        num_patches = seq_len // patch_size
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        self.pos_drop = nn.Dropout(dropout)
        dpr = [x.item() for x in torch.linspace(0, drop_path, depth)]
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, mlp_ratio, dropout, attn_drop, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_embedding(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:, 0]

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        emb = self.forward_embedding(x)
        logits = self.head(emb)
        return logits, emb


# =========================================================
# TRAINING
# =========================================================
@torch.no_grad()
def eval_acc(model, loader):
    model.eval()
    total = 0
    correct = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits, _ = model(x)
        correct += int((logits.argmax(1) == y).sum().item())
        total += y.numel()
    return correct / max(total, 1)


@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    """Run model over a loader and collect (y_true, y_pred) for sklearn metrics."""
    model.eval()
    y_true_list: List[int] = []
    y_pred_list: List[int] = []
    for x, y in loader:
        x = x.to(DEVICE)
        logits, _ = model(x)
        preds = logits.argmax(1).cpu().numpy()
        y_pred_list.extend(preds.tolist())
        y_true_list.extend(y.numpy().tolist())
    return np.array(y_true_list), np.array(y_pred_list)


def train_model() -> RamanViT1D:
    model = RamanViT1D().to(DEVICE)
    counts = np.array([(Y == i).sum() for i in range(NUM_CLASSES)], np.float32)
    weights = torch.tensor(counts.mean() / np.maximum(counts, 1), dtype=torch.float32, device=DEVICE)
    ce = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
                            betas=(0.9, 0.999))

    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS:
            return float(ep + 1) / float(WARMUP_EPOCHS)
        progress = float(ep - WARMUP_EPOCHS) / float(max(1, EPOCHS - WARMUP_EPOCHS))
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_val = -1.0
    best_state = None
    bad = 0
    print("\n" + "=" * 70)
    print("Step 3: Training ViT-1D")
    print("=" * 70)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    for ep in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits, _ = model(x)
            loss = ce(logits, y)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            opt.step()
            losses.append(float(loss.item()))
        sched.step()
        va = eval_acc(model, val_loader)
        print(f"epoch {ep:02d}/{EPOCHS}  loss={np.mean(losses):.4f}  val_acc={va:.3f}  lr={opt.param_groups[0]['lr']:.2e}")
        if va > best_val + 1e-6:
            best_val = va
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= 20:
                print("early stop")
                break
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---- Final report on TEST set ----
    te = eval_acc(model, test_loader)
    print(f"\nBest val acc: {best_val:.4f}")
    print(f"TEST acc:     {te:.4f}")

    # Detailed per-class metrics on TEST set (precision / recall / f1)
    y_true, y_pred = collect_predictions(model, test_loader)
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    print("Confusion matrix (TEST set, rows=true, cols=pred):")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    header = "                  " + "  ".join(f"{c:>14s}" for c in class_names)
    print(header)
    for i, row in enumerate(cm):
        cells = "  ".join(f"{int(v):>14d}" for v in row)
        print(f"  {class_names[i]:16s}  {cells}")

    torch.save(model.state_dict(), MODEL_PATH)
    print("\nSaved model:", MODEL_PATH)
    return model


# =========================================================
# CALIBRATION (centroids + 4 thresholds)
# =========================================================
@torch.no_grad()
def compute_known_centroids(model):
    model.eval()
    embs = {cid: [] for cid in range(NUM_CLASSES)}
    for i in idx_tr:
        x = torch.from_numpy(X[i]).unsqueeze(0).to(DEVICE)
        _, emb = model(x)
        embs[int(Y[i])].append(emb.squeeze(0).cpu().numpy())
    return {cid: np.mean(np.stack(arr), axis=0) for cid, arr in embs.items()}


@torch.no_grad()
def calibrate_thresholds(model, centroids):
    model.eval()
    known_conf, known_margin, known_dist, known_energy = [], [], [], []
    for i in idx_va:
        x = torch.from_numpy(X[i]).unsqueeze(0).to(DEVICE)
        logits, emb = model(x)
        logits_np = logits.squeeze(0).cpu().numpy() / INFER_TEMPERATURE
        prob = softmax_np(logits_np)
        emb_np = emb.squeeze(0).cpu().numpy()
        p_sorted = np.sort(prob)[::-1]
        known_conf.append(float(p_sorted[0]))
        known_margin.append(float(p_sorted[0] - p_sorted[1]))
        known_dist.append(float(min(np.linalg.norm(emb_np - c) for c in centroids.values())))
        known_energy.append(float(np.log(np.sum(np.exp(logits_np)))))

    thr = {
        "conf_min": max(MIN_BLOCK_CONF, percentile_safe(known_conf, 5, 0.7)),
        "margin_min": max(MIN_BLOCK_MARGIN, percentile_safe(known_margin, 5, 0.2)),
        "dist_max": percentile_safe(known_dist, 95, 10.0),
        "energy_min": percentile_safe(known_energy, 5, 0.0),
    }

    print("\n" + "=" * 70)
    print("Step 4: Calibrated thresholds (4-gate open-world rejection)")
    print("=" * 70)
    for k, v in thr.items():
        print(f"  {k:12s} = {v:.4f}")
    return {k: float(v) for k, v in thr.items()}


def save_centroids_and_meta(centroids, thresholds):
    np.savez(CENTROIDS_PATH, **{f"class_{cid}": c for cid, c in centroids.items()})
    meta = {
        "classes": class_names,
        "target_len": TARGET_LEN,
        "patch_size": PATCH_SIZE,
        "num_patches": NUM_PATCHES,
        "drop_n": DROP_N,
        "do_baseline": DO_BASELINE,
        "baseline_window": BASELINE_WINDOW,
        "baseline_smooth": BASELINE_SMOOTH,
        "preprocessing": "v5 (raw-dark -> drop30 -> rolling_min_baseline(15,15) -> min_max_norm) + zero-pad to 2048",
        "split": {
            "train": TRAIN_RATIO,
            "val": VAL_RATIO,
            "test": TEST_RATIO,
            "note": "70/10/20 matching SVM & CNN.",
        },
        "model_architecture": f"ViT-1D (d_model={D_MODEL}, heads={NUM_HEADS}, depth={DEPTH}, patch={PATCH_SIZE})",
        "embed_dim": EMBED_DIM,
        "thresholds": thresholds,
        "raw_min_ptp": RAW_MIN_PTP,
        "raw_min_std": RAW_MIN_STD,
        "infer_temperature": INFER_TEMPERATURE,
        "min_accept_blocks": MIN_ACCEPT_BLOCKS,
        "min_accept_ratio": MIN_ACCEPT_RATIO,
        "min_class_consensus": MIN_CLASS_CONSENSUS,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print("Saved centroids:", CENTROIDS_PATH)
    print("Saved meta:    ", META_PATH)


def load_centroids_and_thresholds():
    if not os.path.exists(CENTROIDS_PATH) or not os.path.exists(META_PATH):
        raise FileNotFoundError("Cannot find saved centroids or meta. Set RETRAIN_MODEL=True and train once first.")
    with open(META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    npz = np.load(CENTROIDS_PATH)
    centroids = {int(k.split("_")[1]): npz[k] for k in npz.files}
    return centroids, meta["thresholds"]


# =========================================================
# PREDICTION
# =========================================================
@torch.no_grad()
def predict_single_file(model, spectrum_path, dark_path, centroids, thresholds, debug=True):
    if not os.path.exists(spectrum_path):
        raise FileNotFoundError(spectrum_path)

    model.eval()
    dark_raw = load_dark_spectrum_from_file(dark_path) if dark_path else np.array([])
    spectra, failed = parse_portable_log_to_spectra(spectrum_path)
    print(f"\n  Parsed: {len(spectra)} spectra, {failed} failed")

    if not spectra:
        return {"matched": False, "label": "UNKNOWN", "reason": "no spectra parsed",
                "num_spectra": 0, "failed_blocks": failed}

    per_block = []
    for idx in range(len(spectra)):
        raw = spectra[idx]
        v, quality = preprocess_single_spectrum_v5(raw, dark_raw)
        if v is None or np.allclose(v, 0):
            if debug:
                print(f"  [BLOCK {idx:03d}] preprocessing failed")
            continue

        if ENABLE_QUALITY_FILTER and (quality["ptp"] < RAW_MIN_PTP or quality["std"] < RAW_MIN_STD):
            if debug:
                print(f"  [BLOCK {idx:03d}] REJECTED (low signal: ptp={quality['ptp']:.1f}, std={quality['std']:.1f})")
            continue

        xt = torch.from_numpy(v).unsqueeze(0).to(DEVICE)
        logits, emb = model(xt)
        logits_np = logits.squeeze(0).cpu().numpy() / INFER_TEMPERATURE
        prob = softmax_np(logits_np)
        pred = int(np.argmax(prob))
        p_sorted = np.sort(prob)[::-1]
        conf = float(p_sorted[0])
        margin = float(p_sorted[0] - p_sorted[1])
        emb_np = emb.squeeze(0).cpu().numpy()
        dists = {cid: float(np.linalg.norm(emb_np - c)) for cid, c in centroids.items()}
        dmin = float(min(dists.values()))
        nearest = min(dists, key=dists.get)
        energy = float(np.log(np.sum(np.exp(logits_np))))

        accept = (conf >= thresholds["conf_min"]
                  and margin >= thresholds["margin_min"]
                  and dmin <= thresholds["dist_max"]
                  and energy >= thresholds["energy_min"])

        per_block.append({
            "idx": idx, "prob": prob, "pred": pred, "conf": conf,
            "margin": margin, "dmin": dmin, "nearest": nearest,
            "energy": energy, "accept": accept,
        })
        if debug:
            print(f"  [BLOCK {idx:03d}] pred={id_to_class[pred]:16s} conf={conf:.4f} "
                  f"margin={margin:.4f} dmin={dmin:.3f} energy={energy:.3f} accept={accept}")

    valid_blocks = len(per_block)
    if valid_blocks == 0:
        return {"matched": False, "label": "UNKNOWN", "reason": "all blocks filtered",
                "num_spectra": len(spectra), "failed_blocks": failed}

    accepted = [b for b in per_block if b["accept"]]
    accepted_blocks = len(accepted)
    required_blocks = max(MIN_ACCEPT_BLOCKS, int(math.ceil(valid_blocks * MIN_ACCEPT_RATIO)))
    file_known_support = np.mean(np.stack([b["prob"] for b in per_block]), axis=0)

    if accepted_blocks < required_blocks:
        label = "UNKNOWN"
        matched = False
        reason = f"accepted_blocks={accepted_blocks} < required={required_blocks}"
    else:
        votes = {}
        for b in accepted:
            name = id_to_class[b["pred"]]
            votes[name] = votes.get(name, 0) + 1
        label = max(votes, key=votes.get)
        consensus = votes[label] / accepted_blocks
        if consensus < MIN_CLASS_CONSENSUS:
            matched = False
            reason = f"class_consensus={consensus:.3f} < {MIN_CLASS_CONSENSUS}"
            label = "UNKNOWN"
        else:
            matched = True
            reason = "accepted known class"

    if debug:
        print("\n" + "=" * 70)
        print(f"  valid blocks   : {valid_blocks}")
        print(f"  failed blocks  : {failed}")
        print(f"  accepted blocks: {accepted_blocks}")
        print(f"  required blocks: {required_blocks}")
        print(f"  FINAL: {label}")
        print(f"  reason: {reason}")
        print(f"  known-class support (averaged over valid blocks):")
        for i in range(NUM_CLASSES):
            print(f"    {id_to_class[i]:16s}  {file_known_support[i]:.4f}")
        print("=" * 70)

    return {
        "matched": matched, "label": label, "reason": reason,
        "num_spectra": len(spectra), "valid_blocks": valid_blocks,
        "failed_blocks": failed, "accepted_blocks": accepted_blocks,
        "required_blocks": required_blocks,
        "known_support": {id_to_class[i]: float(file_known_support[i]) for i in range(NUM_CLASSES)},
    }


# =========================================================
# RUN
# =========================================================
RUNTIME_TEST_FILE, RUNTIME_TEST_DARK = _parse_cli_args()
print("\n" + "=" * 70)
print("Test target")
print("=" * 70)
print(f"  Test file: {RUNTIME_TEST_FILE}")
print(f"  Dark file: {RUNTIME_TEST_DARK}")

if RETRAIN_MODEL or not os.path.exists(MODEL_PATH) or not os.path.exists(CENTROIDS_PATH):
    print("\n>> Training model from scratch...")
    model = train_model()
    centroids = compute_known_centroids(model)
    thresholds = calibrate_thresholds(model, centroids)
    save_centroids_and_meta(centroids, thresholds)
else:
    print("\n>> Loading existing model (set RETRAIN_MODEL=True to retrain)")
    model = RamanViT1D()
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()
    centroids, thresholds = load_centroids_and_thresholds()
    print(f"  Loaded model:     {MODEL_PATH}")
    print(f"  Loaded centroids: {CENTROIDS_PATH}")
    print(f"  Thresholds: {thresholds}")

print("\n" + "=" * 70)
print("Step 5: Inference on test file (with 4-gate open-world rejection)")
print("=" * 70)
res = predict_single_file(model, RUNTIME_TEST_FILE, RUNTIME_TEST_DARK,
                          centroids, thresholds, debug=True)

print("\n" + "=" * 70)
print("FINAL DECISION")
print("=" * 70)
if res["matched"]:
    print(f"  >>> RESULT: {res['label']}")
    print(f"  >>> Reason: {res['reason']}")
else:
    print(f"  >>> RESULT: UNKNOWN")
    print(f"  >>> Reason: {res['reason']}")

print("\n  Known-class support:")
for k, v in sorted(res["known_support"].items(), key=lambda kv: kv[1], reverse=True):
    print(f"    {k:16s}  {v:.4f}")
print("=" * 70)

DEVICE = cpu
MODEL_PATH = C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\_ViT1D_raman_v1\models\vit1d_raman.pt
RETRAIN_MODEL = True
Model: ViT-1D  d_model=128  heads=4  depth=4  patch_size=64  num_tokens=32

Step 1: Load pre-processed training data from .npy files
  r6g                   shape=(113, 2018)
  methylene_blue        shape=(118, 2018)

Total: (231, 2048)  (padded from 2018 to 2048)
Classes: {'r6g': 113, 'methylene_blue': 118}
Value range: [0.0000, 1.0000]

Split ratio: train=0.7  val=0.1  test=0.2
Split count: train=162  val=23  test=46
  r6g                   train=79  val=11  test=23
  methylene_blue        train=83  val=12  test=23

Test target
  Test file: C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\Portable_Raman_database\Database_mythylene_blue.txt
  Dark file: C:\Users\Jeremy666\OneDrive\Documents\laser_test\build\Desktop_Qt_6_6_3_MSVC2019_64bit-Debug\Portable_Raman_databas


Step 3: Training ViT-1D
  Trainable parameters: 812,482


epoch 01/100  loss=0.6945  val_acc=0.522  lr=2.00e-04


epoch 02/100  loss=0.6939  val_acc=0.522  lr=3.00e-04


epoch 03/100  loss=0.6935  val_acc=0.522  lr=4.00e-04


epoch 04/100  loss=0.6939  val_acc=0.522  lr=5.00e-04


... (54 outputs left out)

  [BLOCK 096] pred=methylene_blue   conf=0.9125 margin=0.8250 dmin=0.190 energy=1.408 accept=True
  [BLOCK 097] pred=methylene_blue   conf=0.9124 margin=0.8249 dmin=0.260 energy=1.407 accept=True
  [BLOCK 098] pred=methylene_blue   conf=0.9125 margin=0.8251 dmin=0.109 energy=1.408 accept=True
  [BLOCK 099] pred=methylene_blue   conf=0.9124 margin=0.8248 dmin=0.239 energy=1.407 accept=True
  [BLOCK 100] pred=methylene_blue   conf=0.9127 margin=0.8254 dmin=0.271 energy=1.409 accept=True
  [BLOCK 101] pred=methylene_blue   conf=0.9125 margin=0.8251 dmin=0.096 energy=1.408 accept=True
  [BLOCK 102] pred=methylene_blue   conf=0.9126 margin=0.8252 dmin=0.108 energy=1.408 accept=True


  [BLOCK 103] pred=methylene_blue   conf=0.9125 margin=0.8250 dmin=0.113 energy=1.408 accept=True
  [BLOCK 104] pred=methylene_blue   conf=0.9127 margin=0.8253 dmin=0.144 energy=1.409 accept=True
  [BLOCK 105] pred=methylene_blue   conf=0.9124 margin=0.8248 dmin=0.279 energy=1.407 accept=True
  [BLOCK 106] pred=methylene_blue   conf=0.9125 margin=0.8249 dmin=0.131 energy=1.407 accept=True
  [BLOCK 107] pred=methylene_blue   conf=0.9126 margin=0.8252 dmin=0.111 energy=1.409 accept=True
  [BLOCK 108] pred=methylene_blue   conf=0.9126 margin=0.8251 dmin=0.350 energy=1.408 accept=True


  [BLOCK 109] pred=methylene_blue   conf=0.9125 margin=0.8250 dmin=0.151 energy=1.408 accept=True
  [BLOCK 110] pred=methylene_blue   conf=0.9126 margin=0.8252 dmin=0.295 energy=1.408 accept=True
  [BLOCK 111] pred=methylene_blue   conf=0.9125 margin=0.8250 dmin=0.131 energy=1.408 accept=True
  [BLOCK 112] pred=methylene_blue   conf=0.9125 margin=0.8249 dmin=0.106 energy=1.408 accept=True
  [BLOCK 113] pred=methylene_blue   conf=0.9126 margin=0.8251 dmin=0.160 energy=1.408 accept=True


  [BLOCK 114] pred=methylene_blue   conf=0.9127 margin=0.8254 dmin=0.204 energy=1.409 accept=True
  [BLOCK 115] pred=methylene_blue   conf=0.9127 margin=0.8253 dmin=0.300 energy=1.409 accept=True
  [BLOCK 116] pred=methylene_blue   conf=0.9126 margin=0.8251 dmin=0.075 energy=1.408 accept=True
  [BLOCK 117] pred=methylene_blue   conf=0.9127 margin=0.8255 dmin=0.307 energy=1.409 accept=True

  valid blocks   : 118
  failed blocks  : 11
  accepted blocks: 118
  required blocks: 24
  FINAL: methylene_blue
  reason: accepted known class
  known-class support (averaged over valid blocks):
    r6g               0.0875
    methylene_blue    0.9125

FINAL DECISION
  >>> RESULT: methylene_blue
  >>> Reason: accepted known class

  Known-class support:
    methylene_blue    0.9125
    r6g               0.0875
